# logit-thermo A2-3：T_0 ≈ 0.85 は一般的か — データ集め（GPU、A100）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）
- **ランタイム：A100（メモリ 40GB 以上）**。7B を fp32 で載せるためです。足りない GPU だと ⚠ で止まってランタイムを切ります。見込みは **1〜2 時間**。
- **ゲート `docs/A2_3_GATE.md` v0.2（凍結 2026-10-28 14:15）**。数字を集めて保存するだけで、何も判定しません。結果も表示しません。
- 順番：①自己テスト ②BBC の 40 本（A2-2 と同じことを確かめる）とコードの 40 本を選ぶ（中身は表示しない）③A100 の確かめ（Qwen2.5-1.5B × BBC）→ コード 3 つ → Qwen2.5-7B → OLMo-2-7B。
- 計算のコード：`scripts/a2_2_collect.py`（sha256 `e578cfc480980be6…`）と `scripts/a2_3_collect.py`（`65b16cb5baccaa19…`）をそのまま埋め込み。TF32 は切ります。
- 結果の置き場所：`MyDrive/logit-thermo/A2_3/`。途中で切れても「すべてのセルを実行」で、終わった条件は飛ばして続きから回ります。
- **通知**：ntfy の `YOUR_NTFY_TOPIC` に 🚀・条件ごとの ✅・🎉・⚠️（エラーの文面まるごと）。名札は「logit-thermo A2-3 データ集め（7B 2つ＋コード 3つ＋A100 の確かめ、A100）」。
- **終わったら自動でランタイムを切ります**（止めたいときは最初に `AUTO_DISCONNECT = False`）。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-3 データ集め（7B 2つ＋コード 3つ＋A100 の確かめ、A100）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-3 collect', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 38e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。A100 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
import psutil
if psutil.virtual_memory().total < 50e9:
    notify(f'⚠️ システムの RAM が足りません（{psutil.virtual_memory().total/1e9:.0f} GB）。7B を fp32 で読むのに約 31GB 要ります。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
print('GPU：', torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード（a2_2_collect.py と a2_3_collect.py をそのまま。同じ名前空間で実行）
import hashlib
SRC22 = '"""A2-2（ゲート v0.3）のデータ集め。GPU のノートブックにこのファイルの中身をそのまま埋め込む。\n\n何も判定しない（分析は別の CPU ノートブック）。各モデルについて：\n- 人の文脈（teacher forcing）：各位置・各温度の ℓ（裾の取り分の log-odds）、1 位の log-odds、エントロピー。\n- 温度計の材料：β ＝ 1/T の格子ごとに Σ(⟨z⟩_β − z_y)（人のトークンの平均エネルギーとの差）。\n- a_loc・h（T ＝ 1、人の文脈の各位置）。\n- 生成：各温度・各サンプルで 512 歩、各一歩の ℓ・1 位の log-odds・エントロピーと、選んだトークン。\nすべて float64 で計算し、float32 で保存する。使わない番号（tokenizer の長さ以上）と EOS は −∞ にして外す。\n"""\nimport gc, hashlib, json, math, os, re, time\nimport numpy as np\nimport torch\n\nH = 100\nPROMPT, CONT, N_DOCS, SAMPLES = 64, 512, 40, 2\nTEMPS = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.4, 1.5]\nTCAL_T = np.exp(np.linspace(np.log(0.4), np.log(2.5), 61))   # 温度計の格子（T）\nSEED = 20261027\nMODELS = [\n    (\'pythia-1b\', \'EleutherAI/pythia-1b\', \'Pythia\'),\n    (\'qwen2.5-0.5b\', \'Qwen/Qwen2.5-0.5B\', \'Qwen\'),\n    (\'qwen2.5-0.5b-inst\', \'Qwen/Qwen2.5-0.5B-Instruct\', \'Qwen\'),\n    (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\', \'Qwen\'),\n    (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\', \'OLMo\'),\n    (\'smollm2-1.7b\', \'HuggingFaceTB/SmolLM2-1.7B\', \'SmolLM\'),\n    (\'smollm2-1.7b-inst\', \'HuggingFaceTB/SmolLM2-1.7B-Instruct\', \'SmolLM\'),\n]\nCROSS = [(\'pythia-1b\', \'olmo2-1b\'), (\'qwen2.5-1.5b\', \'smollm2-1.7b\')]\nBBC_MONTHS = [\'2025-04\', \'2025-05\', \'2025-06\']\nBOILER = re.compile(r\'(Getty Images|Image source|Image caption|Related topics|Follow BBC|^Watch:|^Listen:|To play this video you need to enable JavaScript)\')\nH_FRAC = 0.00066   # H を語彙の割合にした版（ゲート §2.1）\n\n\ndef h2_of(n_vocab):\n    return max(2, int(round(H_FRAC * n_vocab)))\n\n\n# ---------- 一歩の量（torch、float64） ----------\n\ndef step_stats(z, T, valid, H2):\n    """z：(B, V) の logit、valid：(V,) の bool。戻り値：ℓ（順位 H 以降）、1 位の log-odds、エントロピー、ℓ（順位 H2 以降）。\n    どれも (B,)、float64。1 位の log-odds は「1 位を除いた logsumexp」から直接計算する（p1 → 1 でも有限）。"""\n    z = z.double().masked_fill(~valid, float(\'-inf\')) / T\n    lse = torch.logsumexp(z, dim=-1)\n    top = torch.topk(z, max(H, H2) - 1, dim=-1)\n\n    def ell_of(h):\n        head = torch.logsumexp(top.values[:, :h - 1], dim=-1)\n        return torch.logsumexp(z.scatter(-1, top.indices[:, :h - 1], float(\'-inf\')), dim=-1) - head\n    rest = torch.logsumexp(z.scatter(-1, top.indices[:, :1], float(\'-inf\')), dim=-1)\n    top1 = top.values[:, 0] - rest\n    lp = z - lse[:, None]\n    p = torch.exp(lp)\n    ent = -torch.where(p > 0, p * lp, torch.zeros_like(p)).sum(-1)\n    return ell_of(H), top1, ent, ell_of(H2)\n\n\ndef np_step_stats(z, T, valid, H2):\n    """自己テスト用の素直な numpy 版（1 行）。"""\n    z = np.where(valid, z.astype(np.float64), -np.inf) / T\n    zs = np.sort(z[np.isfinite(z)])[::-1]\n    def lse(x):\n        m = x.max(); return m + np.log(np.exp(x - m).sum())\n    ell = lse(zs[H - 1:]) - lse(zs[:H - 1])\n    ell2 = lse(zs[H2 - 1:]) - lse(zs[:H2 - 1])\n    lp = zs - lse(zs)\n    return ell, float(zs[0] - lse(zs[1:])), float(-(np.exp(lp) * lp).sum()), ell2\n\n\ndef shape_stats(z, valid):\n    """T ＝ 1 の a_loc（順位 300〜3,000 の傾き）と h（頭の確率 ÷ 裾の確率の log10）。z：(N, V)。"""\n    z = z.double().masked_fill(~valid, float(\'-inf\'))\n    zs = torch.sort(z, dim=-1, descending=True).values\n    r = torch.arange(300, 3001, device=z.device, dtype=torch.float64)\n    x = torch.log(r); x = x - x.mean()\n    y = zs[:, 299:3000]\n    a = -((y - y.mean(-1, keepdim=True)) * x).sum(-1) / (x * x).sum()\n    h = (torch.logsumexp(zs[:, :H - 1], -1) - torch.logsumexp(zs[:, H - 1:], -1)) / math.log(10)\n    return a, h\n\n\ndef tcal_terms(z, y, valid):\n    """温度計の材料：各 T（TCAL_T）で Σ_t(⟨z⟩_T − z_y)。z：(N, V)、y：(N,)。"""\n    z = z.double().masked_fill(~valid, float(\'-inf\'))\n    zy = z.gather(-1, y[:, None])[:, 0]\n    out = []\n    for T in TCAL_T:\n        lp = torch.log_softmax(z / T, dim=-1)\n        p = torch.exp(lp)\n        mean_z = torch.where(p > 0, p * z, torch.zeros_like(p)).sum(-1)\n        out.append(float((mean_z - zy).sum()))\n    return np.array(out)\n\n\n# ---------- 書き出しの文章 ----------\n\ndef clean(text):\n    paras = [p.strip() for p in text.split(\'\\n\')]\n    keep = [p for p in paras if len(p) >= 60 and not BOILER.search(p)]\n    return \'\\n\\n\'.join(keep)\n\n\ndef load_docs():\n    from datasets import load_dataset\n    pool = []\n    for m in BBC_MONTHS:\n        ds = load_dataset(\'RealTimeData/bbc_news_alltime\', m, split=\'train\')\n        for row in ds:\n            c = clean(row[\'content\'] or \'\')\n            if len(c) >= 4000:\n                pool.append({\'month\': m, \'link\': row[\'link\'], \'text\': c})\n    pool.sort(key=lambda d: hashlib.sha256(d[\'link\'].encode()).hexdigest())\n    rng = np.random.default_rng(SEED)\n    idx = rng.choice(len(pool), size=N_DOCS, replace=False)\n    docs = [pool[i] for i in sorted(idx)]\n    meta = {\'pool_size\': len(pool), \'docs\': [{\'month\': d[\'month\'], \'link_sha256\': hashlib.sha256(d[\'link\'].encode()).hexdigest(),\n                                               \'text_sha256\': hashlib.sha256(d[\'text\'].encode()).hexdigest(),\n                                               \'chars\': len(d[\'text\'])} for d in docs]}\n    return docs, meta\n\n\n# ---------- モデルごと ----------\n\ndef load_model(hf_id, device):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    return tok, model\n\n\ndef valid_mask(tok, model, device):\n    V = model.get_output_embeddings().weight.shape[0]\n    valid = torch.zeros(V, dtype=torch.bool, device=device)\n    valid[:len(tok)] = True\n    eos = set()\n    if tok.eos_token_id is not None:\n        eos.add(int(tok.eos_token_id))\n    ge = getattr(model, \'generation_config\', None)\n    if ge is not None and ge.eos_token_id is not None:\n        eos |= set([ge.eos_token_id] if isinstance(ge.eos_token_id, int) else ge.eos_token_id)\n    eot = tok.convert_tokens_to_ids(\'<|endoftext|>\') if \'<|endoftext|>\' in tok.get_vocab() else None\n    if eot is not None:\n        eos.add(int(eot))   # 文書の終わりの記号も外す（SmolLM2-Instruct は EOS が <|im_end|> なので、ここで足す）\n    for e in eos:\n        if e < V:\n            valid[e] = False\n    return valid, sorted(eos)\n\n\ndef split_doc(tok, text):\n    ids = tok(text, add_special_tokens=False).input_ids\n    bos = [tok.bos_token_id] if (tok.bos_token_id is not None and getattr(tok, \'add_bos_token\', False)) else []\n    if len(ids) < PROMPT + CONT:\n        return None\n    return bos + ids[:PROMPT], ids[PROMPT:PROMPT + CONT]\n\n\n@torch.no_grad()\ndef teacher_forced(model, prompt, human, valid, device, H2):\n    x = torch.tensor([prompt + human], device=device)\n    logits = model(x).logits[0, len(prompt) - 1: len(prompt) - 1 + CONT].float()\n    y = torch.tensor(human, device=device)\n    res = {}\n    for k, T in enumerate(TEMPS):\n        res[k] = [v.float().cpu().numpy() for v in step_stats(logits, T, valid, H2)]\n    a, h = shape_stats(logits, valid)\n    return res, a.float().cpu().numpy(), h.float().cpu().numpy(), tcal_terms(logits, y, valid)\n\n\n@torch.no_grad()\ndef generate(model, prompts, T, valid, gen, H2):\n    """prompts：同じ長さのトークン列のリスト。512 歩を KV キャッシュで書く。"""\n    x = torch.tensor(prompts, device=valid.device)\n    out = model(x, use_cache=True)\n    past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    L, T1, E, L2, toks = [], [], [], [], []\n    for t in range(CONT):\n        ell, top1, ent, ell2 = step_stats(logits, T, valid, H2)\n        L.append(ell); T1.append(top1); E.append(ent); L2.append(ell2)\n        z = logits.double().masked_fill(~valid, float(\'-inf\')) / T\n        p = torch.softmax(z, dim=-1)\n        nxt = torch.multinomial(p, 1, generator=gen)\n        toks.append(nxt[:, 0])\n        out = model(nxt, past_key_values=past, use_cache=True)\n        past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    st = lambda xs: torch.stack(xs, 1).float().cpu().numpy()\n    return st(L), st(T1), st(E), st(L2), torch.stack(toks, 1).int().cpu().numpy()\n\n\ndef run_model(name, hf_id, docs, outdir, device, notify=print):\n    path = f\'{outdir}/{name}.npz\'\n    if os.path.exists(path):\n        print(name, \'はもう終わっています（飛ばします）\')\n        return\n    t0 = time.time()\n    tok, model = load_model(hf_id, device)\n    valid, eos = valid_mask(tok, model, device)\n    H2 = h2_of(len(tok))\n    pairs = [split_doc(tok, d[\'text\']) for d in docs]\n    use = [i for i, p in enumerate(pairs) if p is not None]\n    nT, nD = len(TEMPS), len(use)\n    tf = np.zeros((4, nD, nT, CONT), np.float32)   # [ℓ(H), 1 位の log-odds, エントロピー, ℓ(H2)]\n    a_all = np.zeros((nD, CONT), np.float32); h_all = np.zeros((nD, CONT), np.float32)\n    tcal = np.zeros((nD, len(TCAL_T)))\n    for j, i in enumerate(use):\n        res, a, h, tc = teacher_forced(model, *pairs[i], valid, device, H2)\n        for k in range(nT):\n            for q in range(4):\n                tf[q, j, k] = res[k][q]\n        a_all[j], h_all[j], tcal[j] = a, h, tc\n    plen = {len(pairs[i][0]) for i in use}\n    assert len(plen) == 1, plen\n    prompts = [pairs[i][0] for i in use]\n    gen = np.zeros((4, nT, SAMPLES, nD, CONT), np.float32)\n    toks = np.zeros((nT, SAMPLES, nD, CONT), np.int32)\n    for k, T in enumerate(TEMPS):\n        for s in range(SAMPLES):\n            g = torch.Generator(device=device).manual_seed(SEED + 1000 * k + s)\n            l, t1, e, l2, tk = generate(model, prompts, T, valid, g, H2)\n            gen[0, k, s], gen[1, k, s], gen[2, k, s], gen[3, k, s], toks[k, s] = l, t1, e, l2, tk\n        print(f\'  {name} T={T} おわり（{time.time() - t0:.0f} 秒）\')\n    np.savez_compressed(path + \'.part.npz\', tf=tf, a=a_all, h=h_all, tcal=tcal, tcal_T=TCAL_T, gen=gen, toks=toks,\n                        temps=np.array(TEMPS), use=np.array(use), eos=np.array(eos), tok_len=len(tok),\n                        prompt_len=list(plen)[0], revision=str(getattr(model.config, \'_commit_hash\', \'\')),\n                        seconds=time.time() - t0, H2=H2)\n    os.replace(path + \'.part.npz\', path)\n    del model; gc.collect(); torch.cuda.empty_cache()\n    notify(f\'{name} おわり（{time.time() - t0:.0f} 秒）\')\n\n\n@torch.no_grad()\ndef cross_model(reader, writer, docs, outdir, device):\n    """reader が writer の書いた文を読む。文字の窓で、人の続きを読んだときと比べる（記録用）。"""\n    path = f\'{outdir}/cross_{reader}_reads_{writer}.npz\'\n    if os.path.exists(path):\n        return\n    from transformers import AutoTokenizer\n    hf = dict((n, h) for n, h, _ in MODELS)\n    wtok = AutoTokenizer.from_pretrained(hf[writer])\n    W = np.load(f\'{outdir}/{writer}.npz\')\n    tok, model = load_model(hf[reader], device)\n    valid, _ = valid_mask(tok, model, device)\n    H2 = h2_of(len(tok))\n    use = list(W[\'use\']); Wt = W[\'toks\']\n    out = np.full((2, len(TEMPS), SAMPLES, len(use)), np.nan)   # [cross, human] の ℓ の平均\n    for j, i in enumerate(use):\n        text = docs[i][\'text\']\n        enc = wtok(text, add_special_tokens=False, return_offsets_mapping=True)\n        pend = enc[\'offset_mapping\'][PROMPT - 1][1]                  # 書き手の書き出しの終わりの文字位置\n        pre = text[:pend]\n        for k, T in enumerate(TEMPS):\n            for s in range(SAMPLES):\n                gtxt = wtok.decode(Wt[k, s, j])\n                C = min(len(gtxt), len(text) - pend)\n                for q, cont in enumerate([gtxt[:C], text[pend:pend + C]]):\n                    e2 = tok(pre + cont, add_special_tokens=False, return_offsets_mapping=True)\n                    ids = e2[\'input_ids\']; offs = e2[\'offset_mapping\']\n                    bos = [tok.bos_token_id] if (tok.bos_token_id is not None and getattr(tok, \'add_bos_token\', False)) else []\n                    x = torch.tensor([bos + ids], device=device)\n                    logits = model(x).logits[0].float()\n                    pos = [n for n, (a0, _) in enumerate(offs) if pend <= a0 < pend + C and n > 0]\n                    if not pos:\n                        continue\n                    pred = torch.tensor([n - 1 + len(bos) for n in pos], device=device)\n                    ell = step_stats(logits[pred], T, valid, H2)[0]\n                    out[q, k, s, j] = float(ell.mean())\n    np.savez_compressed(path + \'.part.npz\', out=out, temps=np.array(TEMPS))\n    os.replace(path + \'.part.npz\', path)\n    del model; gc.collect(); torch.cuda.empty_cache()\n\n\n# ---------- 自己テスト（本番の前に） ----------\n\n@torch.no_grad()\ndef self_test(device):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    H2 = 33\n    # ① 一歩の量が numpy の素直な計算と一致する（1 位が飛び抜けた行も入れる：1 位の log-odds が有限か）\n    z = torch.randn(3, valid.shape[0], device=device) * 3\n    z[2, 7] += 80.0\n    for T in [0.6, 1.0, 1.5]:\n        got_all = step_stats(z, T, valid, H2)\n        assert all(bool(torch.isfinite(v).all()) for v in got_all), (\'有限でない値\', T)\n        for b in range(3):\n            ref = np_step_stats(z[b].cpu().numpy(), T, valid.cpu().numpy(), H2)\n            got = tuple(float(v[b]) for v in got_all)\n            assert all(abs(g - r) < 1e-6 * max(1, abs(r)) for g, r in zip(got, ref)), (T, got, ref)\n    # ② KV キャッシュの生成と、まとめて計算した logit が一致する\n    g = torch.Generator(device=device).manual_seed(0)\n    prompts = [list(range(5, 5 + 16)), list(range(30, 30 + 16))]\n    x = torch.tensor(prompts, device=device)\n    out = model(x, use_cache=True); past, lg = out.past_key_values, out.logits[:, -1, :]\n    seq, steps = x, []\n    for t in range(8):\n        nxt = torch.multinomial(torch.softmax(lg.masked_fill(~valid, float(\'-inf\')), -1), 1, generator=g)\n        steps.append(lg.clone()); seq = torch.cat([seq, nxt], 1)\n        out = model(nxt, past_key_values=past, use_cache=True); past, lg = out.past_key_values, out.logits[:, -1, :]\n    full = model(seq).logits\n    for t in range(8):\n        assert torch.allclose(full[:, 15 + t, :], steps[t], atol=1e-4), (\'KV キャッシュの不一致\', t)\n    # ③ 1 歩目は生成と人の文脈で同じ分布\n    human = list(range(100, 100 + 64))   # 小さなモデルは位置 512 までなので短くする（使うのは位置 15 だけ）\n    tf_logits = model(torch.tensor([prompts[0] + human], device=device)).logits[0, 15]\n    gen_first = model(torch.tensor([prompts[0]], device=device)).logits[0, -1]\n    assert torch.allclose(tf_logits, gen_first, atol=1e-5), \'1 歩目が一致しない\'\n    return \'OK\'\n'
SRC23 = '"""A2-3（ゲート v0.2、凍結 2026-10-28 14:15）のデータ集め。GPU のノートブックで、scripts/a2_2_collect.py を\n実行したのと同じ名前空間でこのファイルを実行する（A2-2 の関数をそのまま使う）。\n\n足すもの：コードの文章の選び方（ゲート §3）、かたまりに分けた生成（7B のメモリ）、読み込む前のメモリの見積もり、\n記事ごとの NLL。判定はしない（分析は scripts/a2_3_analyze.py）。\n"""\nimport gc, hashlib, json, math, os, re, time\nimport numpy as np\nimport torch\n\nSEED3 = 20261028\nCODE_MONTHS = [\'2025-04\', \'2025-05\', \'2025-06\']\n# 凍結後の修正（2026-10-28 14:3x、データを見る前。分かっていたのは「3 か月では候補が 16 本」だけ）：\n# 足りなければ 2025-03 から過去へ 1 か月ずつ、いちばん古くて 2024-12 まで足す。それでも足りなければ 1 リポジトリ 2 ファイルまで。\n# それでも足りなければ、30 本以上ならその本数で回す。30 本未満なら止める。\nCODE_MONTHS_MORE = [\'2025-03\', \'2025-02\', \'2025-01\', \'2024-12\']\nCODE_MIN = 30\nCODE_EXT = (\'.py\', \'.js\', \'.jsx\', \'.ts\', \'.tsx\', \'.java\', \'.go\', \'.c\', \'.cc\', \'.cpp\', \'.h\', \'.hpp\', \'.rs\', \'.rb\', \'.php\',\n            \'.cs\', \'.kt\', \'.swift\', \'.scala\')\nBAD_PATH = (\'vendor/\', \'node_modules/\', \'dist/\', \'build/\', \'third_party/\', \'.min.\')\nLICENSE_RE = re.compile(r\'(copyright|license|licensed|spdx)\', re.I)\n# 名前, Hugging Face の ID, 文章, 一度に生成する本数\nCONDITIONS = [\n    (\'qwen2.5-1.5b@bbc-a100\', \'Qwen/Qwen2.5-1.5B\', \'bbc\', 40),     # A100 の確かめ（A2-2 と同じ記事・同じ種）\n    (\'pythia-1b@code\', \'EleutherAI/pythia-1b\', \'code\', 40),\n    (\'smollm2-1.7b@code\', \'HuggingFaceTB/SmolLM2-1.7B\', \'code\', 40),\n    (\'qwen2.5-1.5b@code\', \'Qwen/Qwen2.5-1.5B\', \'code\', 40),\n    (\'qwen2.5-7b@bbc\', \'Qwen/Qwen2.5-7B\', \'bbc\', 20),\n    (\'olmo2-7b@bbc\', \'allenai/OLMo-2-1124-7B\', \'bbc\', 8),\n]\n\n\ndef sha(s):\n    return hashlib.sha256(s.encode()).hexdigest()\n\n\ndef code_ok(path, code):\n    """ゲート §3 の残すファイルの条件。"""\n    ext = os.path.splitext(path)[1].lower()\n    if ext not in CODE_EXT:\n        return False\n    p = path.lower()\n    if any(b in p for b in BAD_PATH):\n        return False\n    lines = code.split(\'\\n\')\n    if max(len(l) for l in lines) > 300 or sum(len(l) for l in lines) / len(lines) > 100:\n        return False\n    if LICENSE_RE.search(code[:2000]):\n        return False\n    return len(code) >= 4000\n\n\ndef filter_counts(rows):\n    """条件ごとに何本残るか（記録用）。"""\n    c = {\'rows\': len(rows), \'has_code\': 0, \'ext\': 0, \'path\': 0, \'lines\': 0, \'license\': 0, \'length\': 0}\n    for r in rows:\n        code = (r[\'code\'] or \'\').replace(\'\\r\\n\', \'\\n\'); path = r[\'path\'] or \'\'\n        if not code:\n            continue\n        c[\'has_code\'] += 1\n        if os.path.splitext(path)[1].lower() not in CODE_EXT:\n            continue\n        c[\'ext\'] += 1\n        if any(b in path.lower() for b in BAD_PATH):\n            continue\n        c[\'path\'] += 1\n        lines = code.split(\'\\n\')\n        if max(len(l) for l in lines) > 300 or sum(len(l) for l in lines) / len(lines) > 100:\n            continue\n        c[\'lines\'] += 1\n        if LICENSE_RE.search(code[:2000]):\n            continue\n        c[\'license\'] += 1\n        if len(code) >= 4000:\n            c[\'length\'] += 1\n    return c\n\n\ndef unique_pool(rows, per_repo):\n    pool = []\n    for r in rows:\n        code = (r[\'code\'] or \'\').replace(\'\\r\\n\', \'\\n\')\n        path, repo = r[\'path\'] or \'\', r[\'repo\'] or \'\'\n        if code and code_ok(path, code):\n            pool.append({\'month\': r[\'month\'], \'repo\': repo, \'path\': path, \'text\': code, \'key\': sha(repo + \'\\n\' + path)})\n    pool.sort(key=lambda d: d[\'key\'])\n    uniq, seen_c, per = [], set(), {}\n    for d in pool:\n        c = sha(d[\'text\'])\n        if c in seen_c or per.get(d[\'repo\'], 0) >= per_repo:\n            continue\n        seen_c.add(c); per[d[\'repo\']] = per.get(d[\'repo\'], 0) + 1; uniq.append(d)\n    return pool, uniq\n\n\ndef select_code(rows, per_repo=1, allow_fewer=False):\n    """rows：{\'month\',\'repo\',\'path\',\'code\'} の並び → 選んだ記事（最大 40 本）と目録。中身は返すが、目録にはハッシュだけ。"""\n    pool, uniq = unique_pool(rows, per_repo)\n    if len(uniq) < N_DOCS and not (allow_fewer and len(uniq) >= CODE_MIN):\n        raise RuntimeError(f\'コードの候補が {len(uniq)} 本しかありません（{N_DOCS} 本必要）\')\n    n = min(N_DOCS, len(uniq))\n    rng = np.random.default_rng(SEED3)\n    idx = rng.choice(len(uniq), size=n, replace=False)\n    docs = [uniq[i] for i in sorted(idx)]\n    meta = {\'pool_after_filter\': len(pool), \'pool_unique\': len(uniq), \'per_repo\': per_repo, \'n_docs\': n,\n            \'docs\': [{\'month\': d[\'month\'], \'key_sha256\': d[\'key\'], \'text_sha256\': sha(d[\'text\']),\n                      \'ext\': os.path.splitext(d[\'path\'])[1].lower(), \'chars\': len(d[\'text\'])} for d in docs]}\n    return docs, meta\n\n\ndef load_code_docs():\n    """凍結後の修正の順番どおりに、月を広げ、それでも足りなければ 1 リポジトリ 2 ファイルまで、最後に 30 本以上で回す。"""\n    from datasets import load_dataset\n    rows, months, steps = [], [], []\n    def add(m):\n        for r in load_dataset(\'RealTimeData/code_alltime\', m, split=\'train\'):\n            rows.append({\'month\': m, \'repo\': r[\'repo_name\'], \'path\': r[\'file_path\'], \'code\': r[\'code\']})\n        months.append(m)\n    for m in CODE_MONTHS:\n        add(m)\n    for m in [None] + CODE_MONTHS_MORE:\n        if m is not None:\n            add(m)\n        n1 = len(unique_pool(rows, 1)[1])\n        steps.append({\'months\': list(months), \'unique_per_repo1\': n1})\n        if n1 >= N_DOCS:\n            docs, meta = select_code(rows, 1)\n            break\n    else:\n        n2 = len(unique_pool(rows, 2)[1])\n        steps.append({\'months\': list(months), \'unique_per_repo2\': n2})\n        docs, meta = select_code(rows, 2, allow_fewer=True)\n    meta[\'months\'] = list(months)\n    meta[\'steps\'] = steps\n    meta[\'filter_counts\'] = filter_counts(rows)\n    return docs, meta\n\n\ndef memory_needed(hf_id, chunk):\n    """重み（fp32）＋ KV（fp32、chunk 本 × 577 トークン）＋ 余裕 3GB の見積もり（バイト）。"""\n    from transformers import AutoConfig\n    from huggingface_hub import HfApi\n    cfg = AutoConfig.from_pretrained(hf_id)\n    n_params = None\n    try:\n        info = HfApi().model_info(hf_id, files_metadata=False)\n        if getattr(info, \'safetensors\', None) is not None:\n            n_params = int(info.safetensors.total)\n    except Exception:\n        pass\n    if n_params is None:\n        h, L, V = cfg.hidden_size, cfg.num_hidden_layers, cfg.vocab_size\n        n_params = 12 * L * h * h + 2 * V * h\n    heads = cfg.num_attention_heads\n    kvh = getattr(cfg, \'num_key_value_heads\', None) or heads\n    hd = getattr(cfg, \'head_dim\', None) or cfg.hidden_size // heads\n    kv_tok = 2 * cfg.num_hidden_layers * kvh * hd * 4\n    V = cfg.vocab_size\n    tf_peak = 6 * CONT * V * 8 + (PROMPT + CONT + 1) * V * 4          # 人の文脈の計算の float64 の一時テンソル\n    gen_peak = kv_tok * chunk * (PROMPT + CONT + 1) + chunk * PROMPT * V * 4\n    return int(n_params * 4 + max(tf_peak, gen_peak) + 3e9), int(n_params), int(kv_tok)\n\n\n@torch.no_grad()\ndef teacher_forced3(model, prompt, human, valid, device, H2):\n    """A2-2 の teacher_forced に、記事ごとの NLL（T = 1、外した番号を除いた正規化）を足したもの。"""\n    res, a, h, tc = teacher_forced(model, prompt, human, valid, device, H2)\n    x = torch.tensor([prompt + human], device=device)\n    z = model(x).logits[0, len(prompt) - 1: len(prompt) - 1 + CONT].double().masked_fill(~valid, float(\'-inf\'))\n    y = torch.tensor(human, device=device)\n    nll = float((torch.logsumexp(z, -1) - z.gather(-1, y[:, None])[:, 0]).mean())\n    return res, a, h, tc, nll\n\n\ndef run_condition(name, hf_id, docs, outdir, device, chunk, notify=print):\n    """A2-2 の run_model と同じものを、かたまりに分けた生成・メモリの見積もり・NLL つきで。"""\n    path = f\'{outdir}/{name}.npz\'\n    if os.path.exists(path):\n        print(name, \'はもう終わっています（飛ばします）\')\n        return\n    need, n_params, kv_tok = memory_needed(hf_id, chunk)\n    have = torch.cuda.get_device_properties(0).total_memory\n    if need > have:\n        raise RuntimeError(f\'{name}：GPU のメモリが足りません（見積もり {need / 1e9:.1f} GB、GPU {have / 1e9:.1f} GB）\')\n    t0 = time.time()\n    tok, model = load_model(hf_id, device)\n    assert next(model.parameters()).dtype == torch.float32\n    valid, eos = valid_mask(tok, model, device)\n    H2 = h2_of(len(tok))\n    pairs = [split_doc(tok, d[\'text\']) for d in docs]\n    use = [i for i, p in enumerate(pairs) if p is not None]\n    nT, nD = len(TEMPS), len(use)\n    tf = np.zeros((4, nD, nT, CONT), np.float32)\n    a_all = np.zeros((nD, CONT), np.float32); h_all = np.zeros((nD, CONT), np.float32)\n    tcal = np.zeros((nD, len(TCAL_T))); nll = np.zeros(nD)\n    for j, i in enumerate(use):\n        res, a, h, tc, nl = teacher_forced3(model, *pairs[i], valid, device, H2)\n        for k in range(nT):\n            for q in range(4):\n                tf[q, j, k] = res[k][q]\n        a_all[j], h_all[j], tcal[j], nll[j] = a, h, tc, nl\n    plen = {len(pairs[i][0]) for i in use}\n    assert len(plen) == 1, plen\n    prompts = [pairs[i][0] for i in use]\n    gen = np.zeros((4, nT, SAMPLES, nD, CONT), np.float32)\n    toks = np.zeros((nT, SAMPLES, nD, CONT), np.int32)\n    for k, T in enumerate(TEMPS):\n        for s in range(SAMPLES):\n            for c, lo in enumerate(range(0, nD, chunk)):\n                hi = min(nD, lo + chunk)\n                g = torch.Generator(device=device).manual_seed(SEED + 1000 * k + s + 100000 * c)\n                l, t1, e, l2, tk = generate(model, prompts[lo:hi], T, valid, g, H2)\n                gen[0, k, s, lo:hi], gen[1, k, s, lo:hi], gen[2, k, s, lo:hi], gen[3, k, s, lo:hi] = l, t1, e, l2\n                toks[k, s, lo:hi] = tk\n        print(f\'  {name} T={T} おわり（{time.time() - t0:.0f} 秒）\')\n    ext = np.array([os.path.splitext(docs[i].get(\'path\', \'\'))[1].lower() for i in use])\n    np.savez_compressed(path + \'.part.npz\', tf=tf, a=a_all, h=h_all, tcal=tcal, tcal_T=TCAL_T, gen=gen, toks=toks, nll=nll,\n                        ext=ext, temps=np.array(TEMPS), use=np.array(use), eos=np.array(eos), tok_len=len(tok),\n                        prompt_len=list(plen)[0], revision=str(getattr(model.config, \'_commit_hash\', \'\')),\n                        seconds=time.time() - t0, H2=H2, chunk=chunk, mem_need=need, n_params=n_params, kv_tok=kv_tok)\n    os.replace(path + \'.part.npz\', path)\n    del model; gc.collect(); torch.cuda.empty_cache()\n    notify(f\'{name} おわり（{time.time() - t0:.0f} 秒）\')\n'
assert hashlib.sha256(SRC22.encode()).hexdigest() == 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53'
assert hashlib.sha256(SRC23.encode()).hexdigest() == '65b16cb5baccaa19855f1d23ebb666baaac7c69c9732704307bd297658e6bbc9'
exec(SRC22)
exec(SRC23)
print('コードを置きました')


In [ ]:
# セル3：自己テスト（小さなランダムのモデル）
print('自己テスト：', self_test('cuda'))


In [ ]:
# セル4：文章を選ぶ（中身は表示しない。ここで止まっても、通知してからランタイムを切る）
import json
OUT = '/content/drive/MyDrive/logit-thermo/A2_3'
os.makedirs(OUT, exist_ok=True)
try:
    bbc, bbc_meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in bbc_meta['docs']], 'BBC の記事が A2-2 と違います'
    code_docs, code_meta = load_code_docs()
    mp = OUT + '/code_meta.json'
    if os.path.exists(mp):
        assert [d['text_sha256'] for d in json.load(open(mp))['docs']] == [d['text_sha256'] for d in code_meta['docs']], '前の回とコードが違います'
    else:
        json.dump(code_meta, open(mp, 'w'), indent=1)
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__)).encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 文章を選ぶところで止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise
print('BBC：A2-2 と同じ 40 本。コード：候補', code_meta['pool_unique'], '本から', code_meta['n_docs'], '本（月', code_meta['months'], '、1 リポジトリ', code_meta['per_repo'], 'ファイルまで）')


In [ ]:
# セル5：条件を順に回す（終わった条件は飛ばす）
import transformers, platform
docs_of = {'bbc': bbc, 'code': code_docs}
t_all = time.time()
try:
    for name, hf_id, text, chunk in CONDITIONS:
        run_condition(name, hf_id, docs_of[text], OUT, 'cuda', chunk, notify=lambda m: notify('✅ ' + m, tags='white_check_mark'))
except Exception as e:
    # A100 は高いので、ここでのエラーは通知してからランタイムを切る（終わった条件は保存済みで、続きから回せる）
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__)).encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise
try:
    tf32_rec = torch.get_float32_matmul_precision()
except Exception as e:
    tf32_rec = 'unknown: ' + type(e).__name__
json.dump({'gate': 'docs/A2_3_GATE.md v0.2', 'a2_2_collect_sha256': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'a2_3_collect_sha256': '65b16cb5baccaa19855f1d23ebb666baaac7c69c9732704307bd297658e6bbc9',
           'torch': torch.__version__, 'transformers': transformers.__version__, 'python': platform.python_version(),
           'gpu': torch.cuda.get_device_name(0), 'float32_matmul_precision': tf32_rec,
           'seconds_this_session': round(time.time() - t_all, 1)}, open(OUT + '/run_meta.json', 'w'), indent=1)
print('全部おわり')


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：6 つの条件を Drive に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
